# Open Transport Data Swiss

Download Ist-Daten 2023, 2024, 2025

## Initialer Download

Die Ist-Daten des gesamten Schweizer ÖV werden von [opentransportdata.swiss](https://opentransportdata.swiss) als monatliche ZIP-Archive bereitgestellt.  
Jedes ZIP enthält pro Kalendertag eine CSV-Datei (~100–300 MB), die alle Echtzeit-Ereignisse (Ankünfte, Abfahrten, Verspätungen) für die gesamte Schweiz enthält.

**Zeitraum:** 2023, 2024, 2025 → 36 ZIP-Archive (~50 GB gesamt, ~720 GB entpackt)

**URL-Schema:** `https://archive.opentransportdata.swiss/istdaten/{year}/ist-daten-{year}-{month}.zip`

Das Skript prüft vor jedem Download, ob die Datei bereits lokal vorhanden ist, und überspringt sie in diesem Fall.

In [1]:
import os
import requests

def download_monthly_zips(target_dir="data/otd-swiss/raw"):
    if not os.path.exists(target_dir): os.makedirs(target_dir)
    
    base_url = "https://archive.opentransportdata.swiss/istdaten"
    years = [2023, 2024, 2025]
    
    for year in years:
        for month in range(1, 13):
            month_str = f"{month:02d}"
            # Name der Datei laut deiner Liste
            file_name = f"ist-daten-{year}-{month_str}.zip"
            # Der Pfad ist immer /Jahr/Dateiname
            url = f"{base_url}/{year}/{file_name}"
            
            local_path = os.path.join(target_dir, file_name)
            
            if not os.path.exists(local_path):
                print(f"Lade herunter: {file_name}...")
                r = requests.get(url, stream=True)
                if r.status_code == 200:
                    with open(local_path, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=1024*1024): f.write(chunk)
                else:
                    print(f"Fehler {r.status_code} bei {file_name}")
            else:
                print(f"{file_name} ist schon da.")

# download_monthly_zips()

## Filterung & Parquet-Konvertierung

Da die rohen CSV-Dateien schweizweit alle Betreiber enthalten, werden nur die für dieses Projekt relevanten Zeilen extrahiert:

| Filter | Wert |
| :--- | :--- |
| `BETREIBER_ID` | `85:3849` |
| `PRODUKT_ID` | `Tram` |
| `BETREIBER_ABK` | `VBZ` |


Das Skript `src/process_ist_daten.py` verarbeitet alle ZIP-Archive automatisch:
- Entpackt jedes ZIP, liest jede CSV, filtert VBZ-Tram-Zeilen
- Speichert pro Tag eine `.parquet`-Datei in `data/interim/ist-daten/` (~1–2 MB statt 300 MB)
- Löscht die CSV nach Verarbeitung, ZIPs bleiben als Raw-Backup
- Resume-fähig: bereits vorhandene Parquets werden übersprungen

**Ergebnis:** ~1096 Parquet-Dateien (je ein Tag 2023–2025), gesamt ~600 MB

# Extrahierte Daten

//data/interim/ist-daten/...

 (1,44 GB on disk) for 1.036 items

**Frage:** Polars oder Pandas für den weiteren Workflow?


In [2]:
# Schnellcheck RAM
import psutil
ram_total = psutil.virtual_memory().total / 1e9
ram_free  = psutil.virtual_memory().available / 1e9
print(f"RAM gesamt:    {ram_total:.1f} GB")
print(f"RAM verfügbar: {ram_free:.1f} GB")

RAM gesamt:    17.2 GB
RAM verfügbar: 8.9 GB


In [3]:
# Schnellcheck parquets

from pathlib import Path
print("Arbeitsverzeichnis:", Path.cwd())
PARQUET_DIR = Path("../data/interim/ist-daten/")
files = sorted(PARQUET_DIR.glob("*.parquet"))
print(f"Parquets gefunden: {len(files)}")
print(f"Erste Datei: {files[0].name}")

Arbeitsverzeichnis: /Users/kaywiegand/Workspace/zh-tram-data/notebooks
Parquets gefunden: 1096
Erste Datei: 2023-01-01_istdaten.parquet


### Datenwörterbuch & Spalten-Entscheidung

**Zeilenfilter:**
- Behalten: `AN_PROGNOSE_STATUS = 'REAL'` AND `AB_PROGNOSE_STATUS = 'REAL'` — nur verlässliche Echtzeitmessungen
- Behalten: `FAELLT_AUS_TF = 'true'` — Ausfälle sind der extremste Verspätungsfall
- Entfernen: `DURCHFAHRT_TF = 'true'` — kein Halt, keine relevante Messung
- Entfernen: `ZUSATZFAHRT_TF = 'true'` — kein Fahrplan-Soll, kein sinnvoller Delay

**Feature Engineering:**
- `AN_DELAY` = `AN_PROGNOSE` - `ANKUNFTSZEIT` in Sekunden
- `AB_DELAY` = `AB_PROGNOSE` - `ABFAHRTSZEIT` in Sekunden
- `stop_sequence` = Reihenfolge des Halts innerhalb einer Fahrt (1-basiert), sortiert nach `ANKUNFTSZEIT` pro `FAHRT_BEZEICHNER` + `BETRIEBSTAG`
- Negativ = zu früh, 0 = pünktlich, Positiv = verspätet, NaN = ausgefallen

| Spalte | Entscheidung | Begründung |
| :--- | :--- | :--- |
| `BETRIEBSTAG` | ✅ behalten | Datum für Join mit Events und Wetterdaten |
| `LINIEN_TEXT` | ✅ behalten | Welche Tram-Linie ist betroffen |
| `BPUIC` | ✅ behalten | Join mit GTFS → Koordinaten und Stadtkreis |
| `ANKUNFTSZEIT` | ✅ behalten | Soll-Ankunft — Basis für AN_DELAY und Tageszeit-Analyse |
| `AN_DELAY` | ✅ behalten | Berechnete Verspätung Ankunft in Sekunden |
| `ABFAHRTSZEIT` | ✅ behalten | Soll-Abfahrt — Basis für AB_DELAY und Tageszeit-Analyse |
| `AB_DELAY` | ✅ behalten | Berechnete Verspätung Abfahrt in Sekunden |
| `FAELLT_AUS_TF` | ✅ behalten | Ausfall der gesamten Fahrt — extremster Verspätungsfall |
| `FAHRT_BEZEICHNER` | ✅ behalten als `trip_id` | Fahrt-Identifier — nötig für Trip-Level-Aggregation und Stop-Sequence-Berechnung |
| `BETREIBER_ID` | ❌ entfernt | Nach Filter immer '85:3849' — redundant |
| `BETREIBER_ABK` | ❌ entfernt | Nach Filter immer 'VBZ' — redundant |
| `BETREIBER_NAME` | ❌ entfernt | Nach Filter immer 'Verkehrsbetriebe Zürich' — redundant |
| `PRODUKT_ID` | ❌ entfernt | Nach Filter immer 'Tram' — redundant |
| `LINIEN_ID` | ❌ entfernt | Technischer Schlüssel, LINIEN_TEXT reicht |
| `UMLAUF_ID` | ❌ entfernt | Fahrzeug-ID, für Analyse nicht relevant |
| `VERKEHRSMITTEL_TEXT` | ❌ entfernt | Nach Filter immer 'Tram' — redundant |
| `ZUSATZFAHRT_TF` | ❌ entfernt | Rausgefiltert — kein Fahrplan-Soll vorhanden |
| `HALTESTELLEN_NAME` | ❌ entfernt | Kommt per Join über BPUIC aus GTFS-Daten |
| `AN_PROGNOSE` | ❌ entfernt | In AN_DELAY umgerechnet — Rohdatum nicht mehr nötig |
| `AN_PROGNOSE_STATUS` | ❌ entfernt | Nach Filter immer 'REAL' — redundant |
| `AB_PROGNOSE` | ❌ entfernt | In AB_DELAY umgerechnet — Rohdatum nicht mehr nötig |
| `AB_PROGNOSE_STATUS` | ❌ entfernt | Nach Filter immer 'REAL' — redundant |
| `DURCHFAHRT_TF` | ❌ entfernt | Rausgefiltert — kein Halt, keine relevante Messung |
| `SLOID` | ❌ entfernt | Ersatzfeld für BPUIC, nicht gebraucht |

In [4]:
import pandas as pd
from pathlib import Path

PARQUET_DIR = Path("../data/interim/ist-daten/")
files = sorted(PARQUET_DIR.glob("*.parquet"))

KEEP_COLS = [
    "BETRIEBSTAG",
    "FAHRT_BEZEICHNER",  # trip_id — Fahrt-Identifier, nötig für Trip-Level-Aggregation
    "LINIEN_TEXT",
    "BPUIC",
    "ANKUNFTSZEIT",
    "AN_DELAY",
    "ABFAHRTSZEIT",
    "AB_DELAY",
    "FAELLT_AUS_TF",
    "stop_sequence",     # Reihenfolge des Halts innerhalb einer Fahrt (1-basiert)
]

# Resume-fähig: Rohspalten (z.B. AN_PROGNOSE_STATUS) existieren nur vor der Reduktion.
# Läuft dieser Schritt ein zweites Mal (z.B. nach einer Migration wie hier), sind die
# Parquets bereits auf KEEP_COLS reduziert — dann gibt es nichts mehr zu tun.
sample = pd.read_parquet(files[0])
already_processed = "AN_PROGNOSE_STATUS" not in sample.columns

if already_processed:
    print("Bereits reduziert — Parquets haben nur noch KEEP_COLS (kein Rohspalten-Rest).")
    print(f"Spalten: {list(sample.columns)}")
    stats = {"zeilen_vorher": None, "zeilen_nachher": sum(pd.read_parquet(f).shape[0] for f in files)}
    print(f"Zeilen gesamt: {stats['zeilen_nachher']:,}")
else:
    stats = {"zeilen_vorher": 0, "zeilen_nachher": 0}

    for i, f in enumerate(files):
        df = pd.read_parquet(f)
        stats["zeilen_vorher"] += len(df)

        df_real = df[
            (df["AN_PROGNOSE_STATUS"] == "REAL") &
            (df["AB_PROGNOSE_STATUS"] == "REAL") &
            (df["DURCHFAHRT_TF"]      == "false") &
            (df["ZUSATZFAHRT_TF"]     == "false")
        ]
        df_ausfall = df[
            (df["FAELLT_AUS_TF"]  == "true") &
            (df["DURCHFAHRT_TF"]  == "false") &
            (df["ZUSATZFAHRT_TF"] == "false")
        ]
        df = pd.concat([df_real, df_ausfall], ignore_index=True)

        for col in ["ANKUNFTSZEIT", "AN_PROGNOSE", "ABFAHRTSZEIT", "AB_PROGNOSE"]:
            df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")

        df["AN_DELAY"] = (df["AN_PROGNOSE"] - df["ANKUNFTSZEIT"]).dt.total_seconds()
        df["AB_DELAY"] = (df["AB_PROGNOSE"] - df["ABFAHRTSZEIT"]).dt.total_seconds()

        # stop_sequence: Reihenfolge der Halte innerhalb einer Fahrt (1-basiert)
        # Sortierung nach ANKUNFTSZEIT (Soll), Grupierung nach Fahrt + Betriebstag
        df = df.sort_values(["BETRIEBSTAG", "FAHRT_BEZEICHNER", "ANKUNFTSZEIT"])
        df["stop_sequence"] = (
            df.groupby(["BETRIEBSTAG", "FAHRT_BEZEICHNER"]).cumcount() + 1
        ).astype("int16")

        cols = [c for c in KEEP_COLS if c in df.columns]
        df[cols].to_parquet(f, index=False, engine="pyarrow")
        stats["zeilen_nachher"] += len(df[cols])

        if i % 100 == 0:
            print(f"{i+1}/{len(files)} verarbeitet...")

    print(f"\nFertig!")
    print(f"Zeilen vorher:  {stats['zeilen_vorher']:,}")
    print(f"Zeilen nachher: {stats['zeilen_nachher']:,}")
    print(f"Reduziert um:   {(1 - stats['zeilen_nachher']/stats['zeilen_vorher'])*100:.1f}%")

total_mb = sum(f.stat().st_size for f in files) / 1e6
print(f"Gesamtgrösse:   {total_mb:.0f} MB")


Bereits reduziert — Parquets haben nur noch KEEP_COLS (kein Rohspalten-Rest).
Spalten: ['BETRIEBSTAG', 'FAHRT_BEZEICHNER', 'LINIEN_TEXT', 'BPUIC', 'ANKUNFTSZEIT', 'AN_DELAY', 'ABFAHRTSZEIT', 'AB_DELAY', 'FAELLT_AUS_TF', 'stop_sequence']


Zeilen gesamt: 92,906,148
Gesamtgrösse:   608 MB


**Fertig!**

Zeilen vorher:  98,925,583   
Zeilen nachher: 87,964,096   
Reduziert um:   11.1%   
Gesamtgrösse:   501 MB   

**Zusammenfassung:**

98 Mio → 88 Mio Zeilen — 11% weniger durch die Filter   
1.3 GB → 501 MB — mehr als halbiert durch Spaltenreduktion    

Die 11% rausgefilterten Zeilen sind Durchfahrten, Zusatzfahrten und nicht-REAL Messungen.   


In [5]:
import pandas as pd
import time
import psutil
import os
from pathlib import Path

PARQUET_DIR = Path("../data/interim/ist-daten/")
files = sorted(PARQUET_DIR.glob("*.parquet"))

process = psutil.Process(os.getpid())
ram_before = process.memory_info().rss / 1e9
t0 = time.time()

df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

t1 = time.time()
ram_after = process.memory_info().rss / 1e9

print(f"Zeit:       {t1 - t0:.1f}s")
print(f"RAM delta:  {ram_after - ram_before:.1f} GB")
print(f"Zeilen:     {len(df):,}")
print(f"Spalten:    {df.columns.tolist()}")
print(f"RAM gesamt: {psutil.virtual_memory().available / 1e9:.1f} GB verfügbar")

Zeit:       52.8s
RAM delta:  5.8 GB
Zeilen:     92,906,148
Spalten:    ['BETRIEBSTAG', 'FAHRT_BEZEICHNER', 'LINIEN_TEXT', 'BPUIC', 'ANKUNFTSZEIT', 'AN_DELAY', 'ABFAHRTSZEIT', 'AB_DELAY', 'FAELLT_AUS_TF', 'stop_sequence']
RAM gesamt: 8.8 GB verfügbar


In [6]:
df.head()

,BETRIEBSTAG,FAHRT_BEZEICHNER,LINIEN_TEXT,BPUIC,ANKUNFTSZEIT,AN_DELAY,ABFAHRTSZEIT,AB_DELAY,FAELLT_AUS_TF,stop_sequence
0,01.01.2023,85:3849:100014-05007-1,7,8591106,2023-01-01 21:11:00,34.0,2023-01-01 21:11:00,34.0,false,1
1,01.01.2023,85:3849:100014-05007-1,7,8591279,2023-01-01 21:12:00,34.0,2023-01-01 21:12:00,40.0,false,2
2,01.01.2023,85:3849:100014-05007-1,7,8591304,2023-01-01 21:13:00,58.0,2023-01-01 21:13:00,65.0,false,3
3,01.01.2023,85:3849:100014-05007-1,7,8591081,2023-01-01 21:14:00,58.0,2023-01-01 21:14:00,67.0,false,4
4,01.01.2023,85:3849:100014-05007-1,7,8591082,2023-01-01 21:15:00,58.0,2023-01-01 21:16:00,18.0,false,5


25 Sekunden für 88 Millionen Zeilen — das ist absolut ok für einmal laden am Anfang einer Session!  
Und die Zahlen sind gut:  

6.1 GB RAM für den DataFrame   
6.5 GB noch frei — knapp aber handhabbar   

In [7]:
import polars as pl
import time
import psutil
import os
from pathlib import Path

PARQUET_DIR = Path("../data/interim/ist-daten/")

process = psutil.Process(os.getpid())
ram_before = process.memory_info().rss / 1e9
t0 = time.time()

df = pl.scan_parquet(str(PARQUET_DIR / "*.parquet")).collect()

t1 = time.time()
ram_after = process.memory_info().rss / 1e9

print(f"Zeit:       {t1 - t0:.1f}s")
print(f"RAM delta:  {ram_after - ram_before:.1f} GB")
print(f"Zeilen:     {len(df):,}")
print(f"Spalten:    {df.columns}")
print(f"RAM gesamt: {psutil.virtual_memory().available / 1e9:.1f} GB verfügbar")

Zeit:       10.4s
RAM delta:  0.5 GB
Zeilen:     92,906,148
Spalten:    ['BETRIEBSTAG', 'FAHRT_BEZEICHNER', 'LINIEN_TEXT', 'BPUIC', 'ANKUNFTSZEIT', 'AN_DELAY', 'ABFAHRTSZEIT', 'AB_DELAY', 'FAELLT_AUS_TF', 'stop_sequence']
RAM gesamt: 5.3 GB verfügbar


4x schneller — das ist schon klar.
Aber RAM Delta 0.0 GB ist etwas irreführend — Polars nutzt intern einen anderen Speicher-Allocator der von psutil nicht vollständig erfasst wird.   
Die 5.1 GB verfügbar vs 6.5 GB bei Pandas zeigen dass Polars trotzdem ~1.4 GB belegt — nur anders.  

Fazit:  
Polars ist klar die bessere Wahl für dieses Projekt
4x schneller beim Laden   
Sparsamer im RAM   
Und bei Abfragen und Transformationen wird der Unterschied noch größer  


## Pandas vs. Polars — Benchmark

Getestet auf **87,964,096 Zeilen / 8 Spalten** (501 MB Parquet, 1.035 Dateien).
Operation: alle Parquets laden und als DataFrame bereitstellen.

| | Pandas | Polars |
| :--- | ---: | ---: |
| Ladezeit | 25.7s | 6.6s |
| RAM Delta | 6.1 GB | ~1.4 GB |
| RAM verfügbar danach | 6.5 GB | 5.1 GB |
| Faktor | — | **4x schneller** |

**Entscheidung: Polars**

Polars ist für dieses Projekt die bessere Wahl — deutlich schneller und
speicherschonender. Bei 16 GB RAM und 88 Mio Zeilen ist das kein Nice-to-have
sondern notwendig. Alle weiteren Analysen werden mit Polars durchgeführt.

> Hinweis: Der RAM-Delta von Polars erscheint durch `psutil` als ~0 GB,
> da Polars einen eigenen Speicher-Allocator nutzt. Der tatsächliche
> Verbrauch liegt bei ~1.4 GB (sichtbar an verfügbarem RAM vorher/nachher).

---

## Demo: Einzelner Monat (ausführbar)


## Demo Download: Einzelner Monat (ausführbar)

Die folgenden Zellen zeigen die vollständige Pipeline exemplarisch für den **ersten verfügbaren Tag (01.01.2023)**:  
ZIP herunterladen → entpacken → filtern → als Parquet speichern.

Der gesamte Durchlauf für alle 36 Monate erfolgt über `src/process_ist_daten.py`.

In [8]:
RUN_DEMO = False  # Live-Download vom OTD-Swiss-Archiv — bewusst nicht Teil des
# automatisierten Notebook-Laufs (netzwerk-abhängig, kann lange dauern/hängen).
# Für manuelles, interaktives Ausführen auf True setzen.

if RUN_DEMO:
    import os
    import zipfile
    import requests
    import pandas as pd
    from pathlib import Path

    # --- Pfade ---
    DEMO_DIR = Path("data/demo")
    DEMO_DIR.mkdir(parents=True, exist_ok=True)

    # --- 1. ZIP herunterladen ---
    zip_url  = "https://archive.opentransportdata.swiss/istdaten/2023/ist-daten-2023-01.zip"
    zip_path = DEMO_DIR / "ist-daten-2023-01.zip"

    if not zip_path.exists():
        print("Lade ist-daten-2023-01.zip herunter...")
        r = requests.get(zip_url, stream=True)
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
        print(f"Download abgeschlossen: {zip_path.stat().st_size / 1e6:.0f} MB")
    else:
        print("ZIP bereits vorhanden, überspringe Download.")

    # --- 2. Erste CSV entpacken ---
    extract_dir = DEMO_DIR / "ist-daten-2023-01"
    extract_dir.mkdir(exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        all_csvs = sorted([n for n in zf.namelist() if n.endswith(".csv")])
        first_csv = all_csvs[0]
        zf.extract(first_csv, extract_dir)
        print(f"Entpackt: {first_csv}  ({len(all_csvs)} CSVs im ZIP)")

    csv_path = extract_dir / first_csv

    # --- 3. Filtern: nur VBZ Tram ---
    df = pd.read_csv(csv_path, sep=";", dtype=str, low_memory=False)
    print(f"Gesamtzeilen: {len(df):,}")

    mask = (df["BETREIBER_ID"] == "85:3849") & (df["PRODUKT_ID"] == "Tram")
    df_vbz = df[mask]
    print(f"VBZ Tram-Zeilen: {len(df_vbz):,}")

    # --- 4. Als Parquet speichern ---
    parquet_path = DEMO_DIR / (Path(first_csv).stem + ".parquet")
    df_vbz.to_parquet(parquet_path, index=False, engine="pyarrow")

    csv_size     = csv_path.stat().st_size / 1e6
    parquet_size = parquet_path.stat().st_size / 1e6
    print(f"\nCSV:     {csv_size:.0f} MB")
    print(f"Parquet: {parquet_size:.1f} MB  (Faktor {csv_size/parquet_size:.0f}x kleiner)")
    print(f"Gespeichert: {parquet_path}")

    # --- 5. CSV aufräumen ---
    csv_path.unlink()
    print("CSV gelöscht.")
else:
    print('RUN_DEMO=False — Demo-Download übersprungen (siehe Zellenkopf).')


RUN_DEMO=False — Demo-Download übersprungen (siehe Zellenkopf).
